# 1 · Daten erkunden mit BigQuery
Päckli & Co. verliert Kunden. Eure Aufgabe: Verschafft euch mit SQL einen Überblick.

**Demo:** zeigt die Kursleitung, der Code steht schon da. **Pflicht:** a bis e. **Optional:** f bis h. **🎤** Zu diesen drei Aufgaben wird nach der Übung zufällig eine Person ausgewählt, die ihre Abfrage erklärt.

**Probiert es selbst.** Kommt ihr nicht weiter, fragt Gemini (✦-Knopf unten in der Mitte) oder einen anderen KI-Assistenten eurer Wahl.
Aber: Ihr müsst jede Abfrage eurer Lösung verstehen und erklären können.

## So geht's
- *Datei › Kopie in Drive speichern*, dann im neuen Tab weiterarbeiten.
- Pro Frage: SQL in die leere `%%bigquery`-Zelle schreiben › ▶ oder `Shift + Enter` › **Plausibel?** prüfen.
- Wenn ihr nicht weiterkommt: zuerst den 🧭 Tipp öffnen (zugeklappt: Doppelklick öffnet), dann die KI fragen. Die Lösung gibt es nach der Übung.
- Fehler: ganze Meldung an Gemini, «Erkläre und korrigiere». Eine rosa *Warning* ist kein Fehler.

In [ ]:
#@title ▶ Setup: zuerst ausführen
BASIS_URL = "https://raw.githubusercontent.com/badrustanicki/fhnw-ds-marketing-2026/main/"
import os, sys, urllib.request
try:
    urllib.request.urlretrieve(BASIS_URL + "kurs.py", "kurs.py")   # immer die neuste Version
except OSError:
    if not os.path.exists("kurs.py"): raise
sys.modules.pop("kurs", None)   # neu laden, falls schon geladen
import pandas as pd, matplotlib.pyplot as plt
from kurs import *
print("✅ Bereit.")

### ▶ BigQuery verbinden
Den **Kurs-Schlüssel** einfügen (zeigt die Kursleitung) und ▶. Er gilt nur am Kurstag.

*Nach dem Kurs:* eigenes Gratis-Projekt in [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery) anlegen, dessen Projekt-ID statt des Schlüssels einfügen und im Google-Fenster alle Häkchen setzen. Oder ohne Google: `offline` statt des Schlüssels eintippen, dann laufen die Abfragen über eine Kopie der Daten.

In [ ]:
#@title ▶ BigQuery verbinden
SCHLUESSEL = ""  #@param {type:"string"}
connect_bigquery(SCHLUESSEL, "fhnw-marketing-2026.paeckli")

**Die Daten** im Dataset `fhnw-marketing-2026.paeckli`, in BigQuery immer mit vollem Namen in Backticks: `` `fhnw-marketing-2026.paeckli.orders` ``

| Tabelle | Inhalt |
|---|---|
| `orders` | Rohdaten, eine Zeile = ein Produkt auf einer Rechnung: `invoice`, `stock_code`, `quantity`, `invoice_date`, `price` (CHF), `customer_id` |
| `orders_clean` | dasselbe, bereinigt |
| `products` | `stock_code`, `description` |
| `customers` | `customer_id`, `country` |

Bestellung = Rechnung (`invoice`). Umsatz = `quantity * price`.

### ▶ Erster Blick
Ausführen und lesen.

In [ ]:
display(bq("SELECT * FROM `fhnw-marketing-2026.paeckli.orders` LIMIT 10"))
display(bq("SELECT * FROM `fhnw-marketing-2026.paeckli.products` LIMIT 5"))
bq("SELECT * FROM `fhnw-marketing-2026.paeckli.customers` LIMIT 5")

### Demo: Wie viele Bestellungen gibt es, und wie viele davon sind Stornos?
Wie viele Zeilen, Rechnungen und Kunden hat `fhnw-marketing-2026.paeckli.orders`, und wie viele Rechnungen sind Stornos (Rechnungsnummer beginnt mit C)? Der Code steht schon da: ausführen und lesen.

In [ ]:
%%bigquery
SELECT COUNT(*)                    AS zeilen,
       COUNT(DISTINCT invoice)     AS rechnungen,
       COUNT(DISTINCT customer_id) AS kunden,
       COUNT(DISTINCT IF(invoice LIKE 'C%', invoice, NULL)) AS storno_rechnungen
FROM `fhnw-marketing-2026.paeckli.orders`

**Plausibel?** **Zeilen** oder **Rechnungen** gezählt? Gut 2'700 Rechnungen sind Stornos, kein Umsatz.

### ▶ Bereinigt: orders_clean
Die Rohdaten enthalten Fehler. `orders_clean` entsteht aus `orders` ohne: Stornos, Mengen ≤ 0 oder ab 10'000, Zeilen ohne Kunde, das Testkonto K00001, Porto und Gebühren (POST, DOT, BANK, M, S), doppelte Zeilen und Daten nach dem 31. August 2025. Die Preise im Juni 2024 standen in Rappen und sind in Franken umgerechnet. **Ab hier immer `orders_clean`.** Ausführen: So viel ändert die Bereinigung.

In [ ]:
%%bigquery
SELECT 'orders' AS tabelle, COUNT(*) AS zeilen, ROUND(SUM(quantity * price)) AS umsatz FROM `fhnw-marketing-2026.paeckli.orders`
UNION ALL
SELECT 'orders_clean', COUNT(*), ROUND(SUM(quantity * price)) FROM `fhnw-marketing-2026.paeckli.orders_clean`

### a Wie viele Kunden gibt es pro Land?
**Aufgabe:** Wie viele Kunden gibt es pro Land in `fhnw-marketing-2026.paeckli.customers`? Absteigend sortiert.

In [ ]:
#@title 🧭 Tipp: SQL-Bausteine
# GROUP BY country   eine Zeile pro Land
# COUNT(*)           Kunden pro Gruppe
# ORDER BY ... DESC  absteigend

In [ ]:
%%bigquery
-- Eure Abfrage hier, dann ▶

**Plausibel?** Zusammen 20'001? Einer davon ist das Testkonto K00001.

### b Wie verteilen sich die Kunden auf die Länder, in Prozent?
**Aufgabe:** Wie viel Prozent der Kunden in `fhnw-marketing-2026.paeckli.customers` kommen aus jedem Land? Auf 2 Dezimalstellen gerundet, absteigend.

In [ ]:
#@title 🧭 Tipp: SQL-Bausteine
# Anteil = Anzahl pro Land / Anzahl gesamt * 100
# Gesamtzahl neben jeder Gruppe: SUM(COUNT(*)) OVER ()
# ROUND(wert, 2)

In [ ]:
%%bigquery
-- Eure Abfrage hier, dann ▶

**Plausibel?** Zusammen 100 %?

### c Welches Land bringt den grössten Umsatz? 🎤
**Aufgabe:** Wie gross ist der Umsatz (quantity * price) pro Land? Umsatz in `fhnw-marketing-2026.paeckli.orders_clean`, Land in `fhnw-marketing-2026.paeckli.customers`. Absteigend sortiert.

In [ ]:
#@title 🧭 Tipp: SQL-Bausteine
# Umsatz = SUM(quantity * price)
# Land aus der anderen Tabelle: JOIN customers ON gleiche customer_id
# GROUP BY Land, ORDER BY Umsatz DESC

In [ ]:
%%bigquery
-- Eure Abfrage hier, dann ▶

**Plausibel?** Zusammen rund CHF 9 Mio.? Sonst: `orders` statt `orders_clean` genommen?

### d Wie viele Bestellungen kamen im Juli 2025 pro Land? 🎤
**Aufgabe:** Wie viele Bestellungen (verschiedene Rechnungen) gingen im Juli 2025 pro Land ein? Daten in `fhnw-marketing-2026.paeckli.orders_clean` und `fhnw-marketing-2026.paeckli.customers`. Absteigend sortiert.

In [ ]:
#@title 🧭 Tipp: SQL-Bausteine
# WHERE EXTRACT(YEAR FROM invoice_date) = 2025 AND EXTRACT(MONTH FROM invoice_date) = 7
# Bestellungen = COUNT(DISTINCT invoice), nicht COUNT(*)
# Land über JOIN customers

In [ ]:
%%bigquery
-- Eure Abfrage hier, dann ▶

**Plausibel?** Schweiz rund 2'000? Viel mehr: Zeilen statt Rechnungen gezählt.

### e Wie viele Kunden haben mindestens 10 Mal bestellt, und mit wie viel Umsatz? 🎤
**Aufgabe:** Wie viele Kunden haben in `fhnw-marketing-2026.paeckli.orders_clean` mindestens 10 Bestellungen (verschiedene Rechnungen), und wie gross ist ihr gesamter Umsatz (quantity * price)?

In [ ]:
#@title 🧭 Tipp: SQL-Bausteine
# Pro Kunde rechnen: GROUP BY customer_id
# Gruppen filtern: HAVING COUNT(DISTINCT invoice) >= 10   (WHERE filtert vorher einzelne Zeilen)
# Zusammenfassen: WITH pro_kunde AS (...) SELECT COUNT(*), SUM(umsatz) FROM pro_kunde

In [ ]:
%%bigquery
-- Eure Abfrage hier, dann ▶

**Plausibel?** Rund 1'500 Kunden mit rund zwei Dritteln des Umsatzes? Die wichtigste Gruppe für die Kundenbindung.

## Optional

### f Welcher Anteil der Kunden pro Land ist aktiv?
**Ziel:** Aktiv = Kauf in den 90 Tagen vor dem Stichtag 31. August 2025.

**Aufgabe:** Welcher Anteil der Kunden pro Land hat ab dem 2. Juni 2025 noch gekauft? Daten in `fhnw-marketing-2026.paeckli.orders_clean` und `fhnw-marketing-2026.paeckli.customers`. Pro Land: Kunden, aktive Kunden, Anteil in Prozent.

In [ ]:
#@title 🧭 Tipp: SQL-Bausteine
# WITH: letzter Kauf pro Kunde = MAX(invoice_date), GROUP BY customer_id
# Dann JOIN customers, GROUP BY Land
# Aktive: COUNTIF(zuletzt >= '2025-06-02')

In [ ]:
%%bigquery
-- Eure Abfrage hier, dann ▶

**Plausibel?** Unter einem Drittel aktiv? Darum plant Päckli heute eine Kampagne.

### g Welche 10 Produkte bringen den meisten Umsatz, und wie viele Kunden kauften sie?
**Aufgabe:** Die 10 umsatzstärksten Produkte mit Bezeichnung (`fhnw-marketing-2026.paeckli.products`), Umsatz und Anzahl verschiedener Kunden. Daten in `fhnw-marketing-2026.paeckli.orders_clean`.

In [ ]:
#@title 🧭 Tipp: SQL-Bausteine
# Bezeichnung: JOIN products ON gleicher stock_code
# ORDER BY Umsatz DESC LIMIT 10
# Kunden = COUNT(DISTINCT customer_id)

In [ ]:
%%bigquery
-- Eure Abfrage hier, dann ▶

**Plausibel?** Kein «Porto» dabei? Gut: `orders_clean` enthält nur echte Produkte.

### h Stimmen die Preise in den Rohdaten?
**Ziel:** Durchschnittspreis pro Monat in `orders`.

**Aufgabe:** Zeige den durchschnittlichen Preis pro Monat in `fhnw-marketing-2026.paeckli.orders` (nur quantity > 0), nach Monat sortiert, als Tabelle und als Liniendiagramm.

In [ ]:
#@title 🧭 Tipp: SQL-Bausteine
# Monat: EXTRACT(YEAR FROM invoice_date), EXTRACT(MONTH FROM invoice_date)
# AVG(price), GROUP BY und ORDER BY Jahr, Monat
# Diagramm: Gemini um ein Liniendiagramm bitten

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Ein Monat 100-mal teurer? Rappen statt Franken. Ein Monat im Jahr 2052? Tippfehler. Beides ist in `orders_clean` korrigiert.